# common — executable unit contracts

In [ ]:
from importlib import import_module
from pathlib import Path

import pandas as pd

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file())
unit_files = sorted(path.relative_to(ROOT).as_posix() for path in (ROOT / 'tests/unit').rglob('test_*.py'))

# Every current unit module belongs to one of the five learning notebooks; cases below use production APIs directly.
def lesson(path):
    if any(name in path for name in ('test_dates.py', 'test_manifest.py', 'test_parity.py')): return '04_common.ipynb'
    if 'row_check' in path or 'backends/test_sources.py' in path: return '00_row_check.ipynb'
    if any(name in path for name in ('col_check', 'test_stats.py', 'test_hashing.py', 'test_key_families.py', 'test_sampling_sql.py', 'backends/test_targets.py')): return '01_col_check.ipynb'
    if any(name in path for name in ('dtrack_local/', 'test_protocol_flow.py', 'dtrack_right/report/', 'dtrack_right/test_job.py')): return '02_local.ipynb'
    return '03_tools.ipynb'

coverage = {name: [path for path in unit_files if lesson(path) == name] for name in ('00_row_check.ipynb', '01_col_check.ipynb', '02_local.ipynb', '03_tools.ipynb', '04_common.ipynb')}
assert sorted(path for paths in coverage.values() for path in paths) == unit_files
{name: len(paths) for name, paths in coverage.items()}

In [ ]:
# Date families auto-detect physical formats, then both packages reuse the same portable spec.
for side in ('dtrack_left', 'dtrack_right'):
    dates = import_module(f'{side}.common.dates')
    assert set(dates.DATE_TYPES) == {'date', 'string', 'number', 'sas'}
    assert dates.resolve_date_type('string').detect(['2025-04-02']).spec(portable=True) == {'type': 'string', 'format': 'dash'}
    sas = dates.resolve_date_type('sas').detect(['01AUG2026:09:45:35'])
    assert sas.spec(portable=True) == {'type': 'sas', 'format': 'datetime'}
    assert sas.normalize('01AUG2026:09:45:35') == '2026-08-01'
    assert dates.bucket_date('2026-05-14', 'month') == '2026-05-01'

In [ ]:
# Key normalization and the first eight MD5 hex digits are the cross-platform sampling contract.
from dtrack_left.common.hashing import hash32, key_family, selected_fraction
from dtrack_left.common.sampling import composite_sql, hash32_sql

numeric = key_family('numeric')
assert numeric.normalize('007.50') == '7.5'
assert selected_fraction(['007.50'], ['numeric'], hash32('7.5') + 1)
duckdb_key = composite_sql(['id'], [numeric], 'duckdb', 'left')
spark_key = composite_sql(['`id`'], [numeric], 'spark', 'right')
assert 'md5(' in hash32_sql(duckdb_key, 'duckdb')
assert 'md5(' in hash32_sql(spark_key, 'spark')

In [ ]:
# Manifest windows retain holes and produce stable pair/window/column unit identities.
from dtrack_left.common.manifest import build_manifest, unit_id

pair = {
    'left': {'source': 'file', 'path': 'left.csv', 'date_col': 'D'},
    'right': {'source': 'databricks', 'table': 'right', 'date_col': 'd'},
    'col_map': {'ID': 'id', 'VALUE': 'value'}, 'key': 'ID',
    'key_families': {'ID': 'numeric'}, 'sample': 0.5, 'vintage': 'month',
}
row = {'gate_decision': {'passed': True}, 'partitions': {
    '2026-01-01': {'left': 2, 'right': 2, 'verdict': 'match'},
    '2026-01-02': {'left': 2, 'right': 1, 'verdict': 'count_diff'},
    '2026-01-03': {'left': 2, 'right': 2, 'verdict': 'match'},
}}
manifest = build_manifest('run', 'pair', pair, row)
assert manifest['windows'] == [{'bucket': '2026-01-01', 'partitions': ['2026-01-01', '2026-01-03'], 'n_left': 4}]
assert unit_id('pair', '2026-01-01', 'VALUE') == unit_id('pair', '2026-01-01', 'VALUE')

In [ ]:
# Statistics compare canonical numeric text, strict categorical text, and registered date columns.
from dtrack_left.common.stats import column_statistics
from dtrack_right.checks.col_check import compare_statistics

left = column_statistics(pd.Series(['7.50', '2', None], dtype=object))
right = column_statistics(pd.Series(['7.5', '2.0', None], dtype=object))
assert left['hash_sum'] == right['hash_sum'] and left['n_missing'] == 1
assert compare_statistics(pair='p', window='w', column='amount', left=left, right=right)['verdict'] == 'match'
strict_left = column_statistics(pd.Series(['Gold', 'SILVER']), strict=True)
strict_right = column_statistics(pd.Series(['gold', 'silver']), strict=True)
assert strict_left['hash_sum'] == strict_right['hash_sum']